# Thesis Histograms — Raw | Stage 1 | GNN | CFM | DDPM

Side-by-side per-sample marker histograms for all Stage 2 methods (one row per marker,
one column per method). No training, no GPU — this only loads layers exported by the
explore notebooks.

**Prerequisites** — no Google Drive access is used. Run the export cell in each notebook
first; it downloads the layer files to your computer:
- `spancy_shift_explore.ipynb` → `gnn_*` + `stage1_*` (cell after sec7)
- `spancy_shift_cfm_explore.ipynb` → `cfm_*` (cell 7b)
- `spancy_shift_ddpm_explore.ipynb` → `ddpm_*` (cell 7b)

Then upload them into this runtime at `/content/spancy_layers/` — either drag-and-drop in
the Colab **Files** panel (faster for the ~140 MB `.npy` files) or run the optional upload
cell below. The files are used byte-for-byte as exported (no format conversion).

Missing layers are skipped with a warning, so the figure can be built incrementally.

**Axis convention** (same as the article figure): per marker, bin edges AND y-limit are
derived from **Raw only**, so every column shares identical axes and a peak-height change
is a real shape change, not a binning artifact. Plotted in log1p space.

In [ ]:
!pip install -q anndata
import os, json, shutil
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.backends.backend_pdf import PdfPages
from google.colab import files

# spancy_shift.py is only used for load_adata (same marker names / cell order as the
# explore notebooks) and detect_bimodal_markers (for the [BIMODAL] panel tags).
uploaded = files.upload()
assert os.path.exists('spancy_shift.py'), 'spancy_shift.py not found — upload it'
from spancy_shift import load_adata, detect_bimodal_markers

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

In [ ]:
adata = load_adata('PRAD_anndata.h5ad')
marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X, dtype=np.float32)
X_log_raw = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
obs_names = adata.obs_names.astype(str).to_numpy()

sample_ids = adata.obs['sample_id'].astype(str).values
unique_samples = sorted(np.unique(sample_ids).tolist())
sample_idx = {s: np.where(sample_ids == s)[0] for s in unique_samples}

batch_codes = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')
marker_is_bimodal, _ = detect_bimodal_markers(
    X_log_raw, marker_names,
    batch_codes=batch_codes,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=0.5,
)

print(f'Raw: {X_raw.shape}  ({len(unique_samples)} samples)')
print('Bimodal:', [m for m, b in zip(marker_names, marker_is_bimodal) if b])

In [ ]:
# OPTIONAL — upload the exported layer files (<key>_normalized.npy, <key>_obs_names.npy,
# <key>_meta.json) and move them into EXPORT_DIR. Skip this cell if you already
# drag-and-dropped them into /content/spancy_layers/ via the Files panel (faster).
EXPORT_DIR = '/content/spancy_layers'
os.makedirs(EXPORT_DIR, exist_ok=True)

uploaded_layers = files.upload()
for fname in uploaded_layers:
    shutil.move(fname, os.path.join(EXPORT_DIR, os.path.basename(fname)))
    print(f'-> {EXPORT_DIR}/{os.path.basename(fname)}')

In [ ]:
EXPORT_DIR = '/content/spancy_layers'   # local runtime disk (uploaded files), no Drive

# Column order in the figure (after Raw). Label comes from <key>_meta.json when present.
METHODS = [
    ('stage1', 'Stage 1 (analytic)'),
    ('gnn',    'Stage 2 GNN'),
    ('cfm',    'Stage 2 OT-CFM'),
    ('ddpm',   'Stage 2 DDPM'),
]

layers = {'raw': ('Raw', X_log_raw)}   # key -> (label, log1p matrix)
for key, fallback_label in METHODS:
    path = f'{EXPORT_DIR}/{key}_normalized.npy'
    if not os.path.exists(path):
        print(f'[skip] {key}: {path} not found — download it from its notebook and upload here')
        continue
    arr = np.load(path, mmap_mode='r')
    assert arr.shape == X_raw.shape, f'{key}: shape {arr.shape} != raw {X_raw.shape}'

    # guard against subsample / reorder drift between runtimes
    names_path = f'{EXPORT_DIR}/{key}_obs_names.npy'
    if os.path.exists(names_path):
        assert np.array_equal(np.load(names_path, allow_pickle=True).astype(str), obs_names), \
            f'{key}: cell order differs from raw'
    else:
        print(f'[warn] {key}: no obs_names file — cell order NOT verified (shape only)')

    label = fallback_label
    meta_path = f'{EXPORT_DIR}/{key}_meta.json'
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            label = json.load(f).get('label', fallback_label)

    layers[key] = (label, np.log1p(np.clip(np.asarray(arr), 0, None)).astype(np.float32))
    print(f'[ok]   {key:6s} {label}')

cols = list(layers.keys())
print(f'\nColumns: {[layers[k][0] for k in cols]}')

In [ ]:
N_BINS = 80
cmap = colormaps.get_cmap('tab20')
colors = {s: cmap(i % 20) for i, s in enumerate(unique_samples)}

# Style matches spancy_shift_explore sec7 / the article figure exactly:
# per-sample count curves (tab20, lw 1.5, alpha 0.7), 80 bins, log1p.
# Bin edges AND y-limit per marker from RAW ONLY -> identical axes in every column.
# (Values a method pushes outside the raw range fall outside the bins and are not drawn,
#  same convention as the explore/mxnorm histogram PDFs.)
def _marker_edges(m_idx):
    col = X_log_raw[:, m_idx]
    lo, hi = float(col.min()), float(col.max())
    if not np.isfinite(lo) or hi <= lo:
        hi = lo + 1.0
    return np.linspace(lo, hi, N_BINS + 1)

def _marker_ymax(m_idx, edges):
    col = X_log_raw[:, m_idx]
    ymax = max(int(np.histogram(col[idx], bins=edges)[0].max()) for idx in sample_idx.values())
    return max(ymax, 1)

def plot_marker_row(axes_row, m_idx, show_legend=False):
    edges = _marker_edges(m_idx)
    centers = 0.5 * (edges[:-1] + edges[1:])
    ymax = _marker_ymax(m_idx, edges) * 1.05
    mname = marker_names[m_idx]
    bim_tag = ' [BIMODAL]' if marker_is_bimodal[m_idx] else ''
    for c_idx, (ax, key) in enumerate(zip(axes_row, cols)):
        label, X_log = layers[key]
        col = X_log[:, m_idx]
        for s in unique_samples:
            counts, _ = np.histogram(col[sample_idx[s]], bins=edges)
            ax.plot(centers, counts, color=colors[s], linewidth=1.5, alpha=0.7, label=s)
        ax.set_ylim(0, ymax)
        ax.set_xlim(edges[0], edges[-1])
        ax.set_title(f'{mname}{bim_tag} — {label}', fontsize=9)
        ax.set_xlabel('log1p intensity', fontsize=8)
        if c_idx == 0:
            ax.set_ylabel('Count', fontsize=8)
        if show_legend and c_idx == 0:
            ax.legend(fontsize=5, ncol=3, frameon=False)
        ax.spines[['top', 'right']].set_visible(False)

In [ ]:
# Quick inline check on a few informative markers before writing the full PDF.
PREVIEW_MARKERS = ['ECAD', 'CD45', 'ChromA', 'CD31']

prev = [m for m in PREVIEW_MARKERS if m in marker_names]
fig, axes = plt.subplots(len(prev), len(cols), figsize=(4.2 * len(cols), 3.2 * len(prev)),
                         squeeze=False)
for r, mname in enumerate(prev):
    plot_marker_row(axes[r], marker_names.index(mname), show_legend=(r == 0))
fig.tight_layout()
plt.show()

In [ ]:
ROWS_PER_PAGE = 4
pdf_path = '/content/thesis_histograms_all_methods.pdf'

with PdfPages(pdf_path) as pdf:
    for start in range(0, len(marker_names), ROWS_PER_PAGE):
        page = list(range(start, min(start + ROWS_PER_PAGE, len(marker_names))))
        fig, axes = plt.subplots(len(page), len(cols),   # 7x4 in per panel = same as shift sec7
                                 figsize=(7 * len(cols), 4 * len(page)), squeeze=False)
        for r, m_idx in enumerate(page):
            plot_marker_row(axes[r], m_idx, show_legend=(r == 0))
        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f'PDF saved: {pdf_path}')
files.download(pdf_path)